# Chewing Behaviour Analytics — Integrated Pipeline
**Group 14, Module 2 | INDENG 243**

This notebook covers the full pipeline end-to-end:
1. (Optional) Extract face landmarks from a video
2. Segment bites and compute per-bite features
3. Score health risk against literature thresholds
4. **What-if scenario analysis** — predict how the score changes if behaviour improves

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Make the model module importable from this notebook
MODEL_DIR = Path(".").resolve().parent
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from chewing_health_model import (
    extract_landmarks_from_video,
    segment_bites,
    score_health_risk,
    render_report_text,
    whatif_analysis,
    render_whatif_text,
)

OUT_DIR = MODEL_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)
print("Ready. Output dir:", OUT_DIR)

---
## Step 1 — Video → Landmarks (skip if you already have `mouth_metrics.csv`)

Set `VIDEO_PATH` to your eating video. If you already ran the extraction before,
jump straight to Step 2.

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
VIDEO_PATH  = str(MODEL_DIR / "videos" / "Recording_Dessert.mov")  # ← 视频路径
MODEL_ASSET = str(MODEL_DIR / "models" / "face_landmarker.task")

# 只在没有视频时用这个
CHEWING_CSV    = str(MODEL_DIR / "data" / "chewing_analysis.csv")
TIMESERIES_CSV = str(MODEL_DIR / "outputs" / "mouth_timeseries.csv")
# ───────────────────────────────────────────────────────────────────────────────

if VIDEO_PATH is not None:
    print("[1/4] Extracting landmarks …")
    info = extract_landmarks_from_video(
        VIDEO_PATH, OUT_DIR,
        model_asset_path=MODEL_ASSET,
        max_width=960,          # 3024px → 960px，速度提升约10x
    )
    print(f"  fps={info['fps']:.1f}  frames={info['frame_count']}")
    MOUTH_METRICS_CSV = info["mouth_metrics"]
else:
    print("[1/4] Skipped — using pre-computed data.")
    MOUTH_METRICS_CSV = str(MODEL_DIR / "outputs" / "mouth_metrics.csv")

---
## Step 2 — Bite Segmentation

In [ ]:
if VIDEO_PATH is not None:
    print("[2/4] Segmenting bites …")
    timeseries, bites, fps, auto_gap = segment_bites(MOUTH_METRICS_CSV, OUT_DIR)
else:
    print("[2/4] Loading pre-computed chewing_analysis.csv …")
    bites = pd.read_csv(CHEWING_CSV)
    timeseries = pd.read_csv(TIMESERIES_CSV) if TIMESERIES_CSV else None

print(f"  Bites detected: {len(bites)}")
print(f"  Mean chews / bite: {bites['n_chews'].mean():.2f}")
print(f"  Mean frequency Hz: {bites['chewing_frequency_per_sec'].mean():.3f}")
print(f"  Mean duration sec: {bites['duration_sec'].mean():.2f}")
print(f"  Chew side counts:\n{bites['chew_side'].value_counts().to_string()}")
bites.head()

In [ ]:
# ── Visualise mouth signal (only when we have the time series) ─────────────────
if timeseries is not None and "mouth_open_smooth" in timeseries.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    ts = timeseries
    axes[0].plot(ts["time_sec"], ts["mouth_open_px"], color="lightgrey", lw=0.7, label="raw")
    axes[0].plot(ts["time_sec"], ts["mouth_open_smooth"], color="steelblue", lw=1.2, label="smoothed")
    axes[0].set_ylabel("Mouth open (px)"); axes[0].legend(fontsize=8)
    axes[0].set_title("Mouth Opening Signal")

    axes[1].fill_between(ts["time_sec"], ts["mouth_is_open"],
                         alpha=0.4, color="tomato", label="mouth open")
    axes[1].set_ylabel("Open / Closed"); axes[1].set_xlabel("Time (s)")
    axes[1].set_title("Detected Open Phases")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "mouth_signal.png", dpi=150)
    plt.show()
else:
    print("No timeseries available — skipping signal plot.")

---
## Step 3 — Health Risk Scoring

In [ ]:
print("[3/4] Scoring health risk …")
report = score_health_risk(bites)
print(render_report_text(report))

# Save
(OUT_DIR / "health_report.json").write_text(json.dumps(report.to_dict(), indent=2))
(OUT_DIR / "health_report.txt").write_text(render_report_text(report))

In [ ]:
# ── Score breakdown bar chart ──────────────────────────────────────────────────
features   = [f.feature for f in report.per_feature]
pts        = [f.points for f in report.per_feature]
max_pts    = [f.max_points for f in report.per_feature]
categories = [f.category for f in report.per_feature]
pct        = [100 * p / m for p, m in zip(pts, max_pts)]

colors = {"Metabolic/Obesity": "tomato", "Oral/TMJ": "steelblue", "Behavioural": "goldenrod"}
bar_colors = [colors.get(c, "grey") for c in categories]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(features, pct, color=bar_colors, edgecolor="k", height=0.5)
ax.set_xlim(0, 100)
ax.axvline(60, color="red", ls="--", lw=1, label="HIGH threshold (60%)")
ax.axvline(30, color="orange", ls="--", lw=1, label="MODERATE threshold (30%)")
for bar, p in zip(bars, pct):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{p:.0f}%", va="center", fontsize=9)
patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=patches + ax.get_legend_handles_labels()[0][2:], fontsize=8, loc="lower right")
ax.set_xlabel("Risk contribution (%)")
ax.set_title(f"Health Risk Score per Feature  —  Overall: {report.overall_risk_pct}% ({report.risk_level})")
plt.tight_layout()
plt.savefig(OUT_DIR / "score_breakdown.png", dpi=150)
plt.show()

---
## Step 4 — What-If Scenario Analysis

**Idea:** given the current score, what would happen if specific chewing behaviours changed?

Edit the `scenarios` list below to test your own hypotheses.

Available override keys:
| Key | Meaning |
|---|---|
| `chews_per_bite` | mean chews per bite (e.g. 30) |
| `chewing_frequency_hz` | mean chew rate in Hz (e.g. 1.0 = slow) |
| `dominant_side_share` | fraction of bites on one side (0.5 = balanced) |
| `bite_duration_sec` | mean bite duration in seconds (e.g. 4.0) |

In [ ]:
# ── Define your scenarios ──────────────────────────────────────────────────────
scenarios = [
    {"name": "Chew 20x per bite",    "chews_per_bite": 20},
    {"name": "Chew 30x per bite",    "chews_per_bite": 30},
    {"name": "Chew 40x per bite",    "chews_per_bite": 40},
    {"name": "Eat slower (1.0 Hz)",  "chewing_frequency_hz": 1.0},
    {"name": "Eat slower (0.8 Hz)",  "chewing_frequency_hz": 0.8},
    {"name": "Balanced sides (55%)", "dominant_side_share": 0.55},
    {"name": "Longer bites (4s)",    "bite_duration_sec": 4.0},
    {
        "name": "All improvements",
        "chews_per_bite": 30,
        "chewing_frequency_hz": 1.0,
        "dominant_side_share": 0.55,
        "bite_duration_sec": 4.0,
    },
]

print("[4/4] Running what-if analysis …")
wi_results = whatif_analysis(bites, scenarios, baseline_report=report)
print(render_whatif_text(report, wi_results))

(OUT_DIR / "whatif_report.txt").write_text(render_whatif_text(report, wi_results))
(OUT_DIR / "whatif_report.json").write_text(json.dumps([
    {"scenario": r.scenario_name, "changes": r.changes,
     "risk_pct": r.risk_pct, "risk_level": r.risk_level,
     "delta_risk_pct": r.delta_risk_pct, "per_category": r.per_category}
    for r in wi_results
], indent=2))

In [ ]:
# ── What-if visualisation ──────────────────────────────────────────────────────
names      = [r.scenario_name for r in wi_results]
risk_pcts  = [r.risk_pct for r in wi_results]
deltas     = [r.delta_risk_pct for r in wi_results]
bar_cols   = ["tomato" if d >= 0 else "seagreen" for d in deltas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute risk per scenario vs baseline
ax = axes[0]
y = range(len(names))
ax.barh(y, risk_pcts, color="steelblue", edgecolor="k", height=0.5, label="Scenario risk")
ax.axvline(report.overall_risk_pct, color="red", ls="--", lw=1.5,
           label=f"Baseline {report.overall_risk_pct}%")
ax.axvline(60, color="orange", ls=":", lw=1, label="HIGH threshold")
ax.axvline(30, color="gold", ls=":", lw=1, label="MODERATE threshold")
ax.set_yticks(list(y)); ax.set_yticklabels(names, fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("Overall Risk Score (%)")
ax.set_title("Absolute Risk per Scenario")
ax.legend(fontsize=8)

# Right: delta vs baseline
ax = axes[1]
ax.barh(list(y), deltas, color=bar_cols, edgecolor="k", height=0.5)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(list(y)); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Change in Risk Score (pp)  [green = improvement]")
ax.set_title("Risk Change vs Baseline")
for i, d in enumerate(deltas):
    ax.text(d + (0.5 if d >= 0 else -0.5), i,
            f"{d:+.1f}%", va="center", ha="left" if d >= 0 else "right", fontsize=8)

plt.suptitle("What-If Scenario Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "whatif_chart.png", dpi=150)
plt.show()

---
## Custom scenario — try your own
Change the values below and re-run the cell.

In [ ]:
custom = [
    {
        "name": "My custom scenario",
        # ← change any of these values:
        "chews_per_bite": 25,
        "chewing_frequency_hz": 1.2,
        # "dominant_side_share": 0.60,
        # "bite_duration_sec": 3.0,
    }
]
custom_results = whatif_analysis(bites, custom, baseline_report=report)
print(render_whatif_text(report, custom_results))

---
## Outputs summary

| File | Description |
|---|---|
| `outputs/health_report.txt` | Current behaviour health report |
| `outputs/health_report.json` | Machine-readable version |
| `outputs/score_breakdown.png` | Per-feature risk bar chart |
| `outputs/whatif_report.txt` | What-if scenario results |
| `outputs/whatif_report.json` | Machine-readable what-if results |
| `outputs/whatif_chart.png` | Scenario comparison chart |